In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from visualisation import *
from plotly.subplots import make_subplots

Create some data

In [ ]:
from generate_data import gen_2d_data
from diffusion_geometry.classes.main import DiffusionGeometry

data1 = gen_2d_data(index = 2, n = 300, noise = 0.08, seed = 0)[0]
data1 = data1[:, [1, 0]]
data1 *= [1, 1]
plot_scatter_2d(data1, color = 'black').show()
dg1 = DiffusionGeometry.from_point_cloud(data1)

In [ ]:
# Up Hodge energy <da,db>
UH1 = dg1.up_laplacian(1)
DH1 = dg1.down_laplacian(1)

# Hodge energy <da,db> + <d^*a,d^*b>
H1 = 1e-5 * UH1 + DH1
vals_1, forms_1 = H1.spectrum()

quiver_scale = 0.1
arrow_scale = 0.3
line_width = 1.5

X = forms_1[0].sharp()
f11 = plot_quiver_2d(data1, X.to_ambient(), scale=quiver_scale, arrow_scale=arrow_scale, line_width=line_width)
f11.show()

Y = dg1.function(data1[:,1]).grad()
f12 = plot_quiver_2d(data1, Y.to_ambient(), scale=quiver_scale, arrow_scale=arrow_scale, line_width=line_width)
f12.show()

norm_X = X.pointwise_norm()
f13 = plot_scatter_2d(data1, norm_X, size = 4)
f13.show()

gXY = dg1.g(X, Y)
f14 = plot_scatter_2d(data1, gXY, size = 4)

norm_Y = Y.pointwise_norm()
f15 = plot_scatter_2d(data1, norm_Y, size = 4)


In [ ]:
from generate_data import gen_3d_data

data2 = gen_3d_data(kind = 'torus', minor_radius = 1.0, major_radius=2.0, n = 1000, noise = 0.0, seed = 0)[0]
data2 = data2[:, [1, 2, 0]]


camera2 = dict(eye=dict(x=2, y=1.5, z=0.3),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1))


fig = plot_scatter_3d(data2, color = 'black')
fig.update_layout(scene_camera=camera2)
fig.update_layout(width=400, height=450)
fig.show()

dg2 = DiffusionGeometry.from_point_cloud(data2)

In [ ]:
# Up Hodge energy <da,db>
UH1 = dg2.up_laplacian(1)
DH1 = dg2.down_laplacian(1)

# Hodge energy <da,db> + <d^*a,d^*b>
H1 = 1e-2 * UH1 + DH1
vals_1, forms_1 = H1.spectrum()

X = forms_1[0].sharp()
f21 = plot_quiver_3d(data2, X.to_ambient(), scale = 0.3, arrow_scale=1.2, line_width=3)
f21.update_layout(scene_camera=camera2)
f21.show()

Y = dg2.function(data2[:,0]).grad()
f22 = plot_quiver_3d(data2, Y.to_ambient(), scale = 0.3, arrow_scale=1.2, line_width=3)
f22.update_layout(scene_camera=camera2)
f22.show()

norm_X = X.pointwise_norm()
f23 = plot_scatter_3d(data2, norm_X, size = 2)
f23.update_layout(scene_camera=camera2)
f23.show()

gXY = dg2.g(X, Y)
f24 = plot_scatter_3d(data2, gXY, size = 2)
f24.update_layout(scene_camera=camera2)
f24.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Define which subplots are 3D
specs = [
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    rows=2,
    cols=4,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.01,
)

# Add traces from each figure
fig.add_traces(list(f11.data), rows=[1]*len(f11.data), cols=[1]*len(f11.data))
fig.add_traces(list(f13.data), rows=[1]*len(f13.data), cols=[2]*len(f13.data))
fig.add_traces(list(f12.data), rows=[1]*len(f12.data), cols=[3]*len(f12.data))
fig.add_traces(list(f14.data), rows=[1]*len(f14.data), cols=[4]*len(f14.data))

fig.add_traces(list(f21.data), rows=[2]*len(f21.data), cols=[1]*len(f21.data))
fig.add_traces(list(f23.data), rows=[2]*len(f23.data), cols=[2]*len(f23.data))
fig.add_traces(list(f22.data), rows=[2]*len(f22.data), cols=[3]*len(f22.data))
fig.add_traces(list(f24.data), rows=[2]*len(f24.data), cols=[4]*len(f24.data))

# Synchronize axis ranges for top row (2D plots)
all_figs = [f11, f12, f13, f14]
x_vals = []
y_vals = []
for f in all_figs:
    for t in f.data:
        if hasattr(t, 'x') and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, 'y') and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])

xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]

# Apply to all top row subplots
fig.update_xaxes(range=xrange, row=1)
fig.update_yaxes(range=yrange, row=1)

# Synchronize axis ranges for bottom row (3D plots)
all_figs_3d = [f21, f22, f23, f24]
x_vals_3d = []
y_vals_3d = []
z_vals_3d = []
for f in all_figs_3d:
    for t in f.data:
        if hasattr(t, 'x') and t.x is not None:
            x_vals_3d.extend([v for v in t.x if v is not None])
        if hasattr(t, 'y') and t.y is not None:
            y_vals_3d.extend([v for v in t.y if v is not None])
        if hasattr(t, 'z') and t.z is not None:
            z_vals_3d.extend([v for v in t.z if v is not None])

xrange_3d = [min(x_vals_3d), max(x_vals_3d)]
yrange_3d = [min(y_vals_3d), max(y_vals_3d)]
zrange_3d = [min(z_vals_3d), max(z_vals_3d)]

# Apply to all bottom row 3D subplots
for i in range(1, 5):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        key: dict(
            camera=camera2,
            xaxis=dict(range=xrange_3d),
            yaxis=dict(range=yrange_3d),
            zaxis=dict(range=zrange_3d)
        )
    })

fig.update_layout(width=1000, height=800)
clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

fig.show()
fig.write_image("figs/new_output.pdf", scale=1)


In [ ]:

rows, cols = 2, 4
fig_w, fig_h = 1000, 800

def labels(row, col):
    if col == 1:
        return rf"$X$"
    elif col == 2:
        return "$\|X\| = \sqrt{g(X,X)}$"
    elif col == 3:
        return rf"$Y$"
    elif col == 4:
        return rf"$g(X,Y)$"
    else:
        return ""

overpic_labels(fig, labels, 
               stretch_x=0.98, 
               stretch_y=0.96,
               offset_y=-1.0)


In [ ]:
from figures.generate_data import load_image_point_cloud

data3 = load_image_point_cloud("../data/data1.jpeg", n=800, threshold=0.5, intensity_weighted=True)
data3 = data3[:, [1, 0]]

plot_scatter_2d(data3, color='black').show()

In [ ]:
dg3 = DiffusionGeometry.from_point_cloud(data3)

X = dg3.form_space(2).zeros()
X.coeffs[1] = 1
f31 = plot_2form_2d(data3, X.to_ambient())
f31.show()

Y = dg3.form_space(2).zeros()
Y.coeffs[4] = 1
f32 = plot_2form_2d(data3, Y.to_ambient())
f32.show()

norm_X = X.pointwise_norm()
f33 = plot_scatter_2d(data3, norm_X, size = 5)
f33.show()

gXY = dg3.g(X, Y)
f34 = plot_scatter_2d(data3, gXY, size = 5)
f34.show()


In [ ]:
data4 = gen_3d_data(kind = 'torus', minor_radius = 1.0, major_radius=2.0, n = 2000, noise = 0.0, seed = 0)[0]
data4 = data4[:, [1, 2, 0]]

k = 0.9
camera2_scaled = dict(eye=dict(x=camera2['eye']['x']*k, y=camera2['eye']['y']*k, z=camera2['eye']['z']*k),
                center=dict(x=camera2['center']['x'], y=camera2['center']['y']-0.05, z=camera2['center']['z']+0.07),
                up=camera2['up'])


fig = plot_scatter_3d(data4, color = 'black', camera=camera2_scaled)
fig.update_layout(scene_camera=camera2_scaled)
fig.update_layout(width=400, height=450)
fig.show()

dg4 = DiffusionGeometry.from_point_cloud(data4)

In [ ]:
# Up Hodge energy <da,db>
UH2 = dg4.up_laplacian(2)
DH2 = dg4.down_laplacian(2)

# Hodge energy <da,db> + <d^*a,d^*b>
H2 = 1e-2 * UH2 + DH2
vals_2, forms_2 = H2.spectrum()

X = dg4.form_space(2).zeros()
X.coeffs[0] = 1
f41 = plot_2form_3d(data4, X.to_ambient())
f41.update_layout(scene_camera=camera2_scaled)
f41.show()

Y = forms_2[0]
f42 = plot_2form_3d(data4, Y.to_ambient())
f42.update_layout(scene_camera=camera2_scaled)
f42.show()

norm_X = X.pointwise_norm()
f43 = plot_scatter_3d(data4, norm_X, size = 3)
f43.update_layout(scene_camera=camera2_scaled)
f43.show()

gXY = dg4.g(X, Y)
f44 = plot_scatter_3d(data4, gXY, size = 3)
f44.update_layout(scene_camera=camera2_scaled)
f44.show()


In [ ]:
data5, _ = gen_3d_data("cone", n=2000, noise=0.0, height=1.0, radius=0.4, solid=True)
dg5 = DiffusionGeometry.from_point_cloud(data5)

camera5 = dict(eye=dict(x=1.4, y=1.0, z=0.8),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1))

fig = plot_scatter_3d(data5, color='black')
fig.update_layout(scene_camera=camera5)
# fig.update_layout(width=400, height=450)
fig.show()

In [ ]:
X = dg5.form_space(3).zeros()
X.coeffs[1] = 1
f51 = plot_3form_3d(data5, X.to_ambient(), camera=camera5)
f51.update_layout(scene_camera=camera5)
f51.show()

Y = dg5.form_space(3).zeros()
Y.coeffs[2] = 1
f52 = plot_3form_3d(data5, Y.to_ambient(), camera=camera5)
f52.update_layout(scene_camera=camera5)
f52.show()

norm_X = X.pointwise_norm()
f53 = plot_scatter_3d(data5, norm_X, size = 3)
f53.update_layout(scene_camera=camera5)
f53.show()

gXY = dg5.g(X, Y)
f54 = plot_scatter_3d(data5, gXY, size = 3)
f54.update_layout(scene_camera=camera5)
f54.show()


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

figure_rows = [
    [f31, f33, f32, f34],
    [f41, f43, f42, f44],
    [f51, f53, f52, f54],
]


def _is_3d_figure(fig_obj):
    for trace in fig_obj.data:
        trace_type = getattr(trace, "type", "") or ""
        if "3d" in trace_type.lower():
            return True
        if hasattr(trace, "z") and trace.z is not None:
            return True
    return False


specs = []
scene_map = {}
scene_counter = 0
for row_idx, row_figs in enumerate(figure_rows, start=1):
    row_is_3d = any(_is_3d_figure(fig_obj) for fig_obj in row_figs)
    row_type = "scene" if row_is_3d else "xy"
    specs.append([{"type": row_type} for _ in row_figs])
    if row_type == "scene":
        for col_idx in range(1, len(row_figs) + 1):
            scene_counter += 1
            scene_name = "scene" if scene_counter == 1 else f"scene{scene_counter}"
            scene_map[(row_idx, col_idx)] = scene_name

fig = make_subplots(
    rows=len(figure_rows),
    cols=len(figure_rows[0]),
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.01,
)

for row_idx, row_figs in enumerate(figure_rows, start=1):
    for col_idx, src_fig in enumerate(row_figs, start=1):
        traces = list(src_fig.data)
        if not traces:
            continue
        fig.add_traces(
            traces,
            rows=[row_idx] * len(traces),
            cols=[col_idx] * len(traces),
        )

for row_idx, row_figs in enumerate(figure_rows, start=1):
    row_type = specs[row_idx - 1][0]["type"]
    if row_type == "xy":
        x_vals, y_vals = [], []
        for src_fig in row_figs:
            for trace in src_fig.data:
                if hasattr(trace, "x") and trace.x is not None:
                    x_vals.extend(v for v in trace.x if v is not None)
                if hasattr(trace, "y") and trace.y is not None:
                    y_vals.extend(v for v in trace.y if v is not None)
        if x_vals and y_vals:
            fig.update_xaxes(range=[min(x_vals), max(x_vals)], row=row_idx)
            fig.update_yaxes(range=[min(y_vals), max(y_vals)], row=row_idx)
    else:
        x_vals, y_vals, z_vals = [], [], []
        for src_fig in row_figs:
            for trace in src_fig.data:
                if hasattr(trace, "x") and trace.x is not None:
                    x_vals.extend(v for v in trace.x if v is not None)
                if hasattr(trace, "y") and trace.y is not None:
                    y_vals.extend(v for v in trace.y if v is not None)
                if hasattr(trace, "z") and trace.z is not None:
                    z_vals.extend(v for v in trace.z if v is not None)
        if x_vals and y_vals and z_vals:
            x_range = [min(x_vals), max(x_vals)]
            y_range = [min(y_vals), max(y_vals)]
            z_range = [min(z_vals), max(z_vals)]
            row_camera = None
            for src_fig in row_figs:
                scene_layout = getattr(src_fig.layout, "scene", None)
                if scene_layout and scene_layout.camera:
                    row_camera = scene_layout.camera
                    break
            if row_camera is None:
                row_camera = globals().get("camera")
            for col_idx in range(1, len(row_figs) + 1):
                scene_name = scene_map[(row_idx, col_idx)]
                scene_update = dict(
                    xaxis=dict(range=x_range),
                    yaxis=dict(range=y_range),
                    zaxis=dict(range=z_range),
                )
                if row_camera is not None:
                    scene_update["camera"] = row_camera
                fig.update_layout({scene_name: scene_update})

fig.update_layout(width=1000, height=1000)
clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

fig.show()
fig.write_image("figs/new_output_f345.pdf", scale=1)


In [ ]:
def labels(row, col):
    if col == 1:
        return rf"$\alpha$"
    elif col == 2:
        return rf"$\|\alpha\| = \sqrt{{g(\alpha,\alpha)}}$"
    elif col == 3:
        return rf"$\beta$"
    elif col == 4:
        return rf"$g(\alpha,\beta)$"
    else:
        return ""

overpic_labels(fig, labels, 
               stretch_x=0.98, 
               stretch_y=0.93,
               offset_y=16.5,
               offset_x=-0.)
